# THỰC HÀNH THU THẬP DỮ LIỆU WEB BẰNG PYTHON

## Mục tiêu học tập
Web Scraping, Web Data Formats và Selenium
1. Giải thích web scraping là gì và khi nào nên dùng.
2. Hiểu cơ chế HTTP Request/Response.
3. Đọc cấu trúc HTML, tag, attribute, class, id và table.
4. Dùng `requests` để tải nội dung web.
5. Dùng `BeautifulSoup` để parse HTML.
6. Thu thập dữ liệu từ một trang và nhiều trang.
7. Tổ chức dữ liệu thu thập bằng Pandas.
8. Đọc dữ liệu XML và JSON.
9. Hiểu XPath và Selenium.
10. Xây dựng crawler nhỏ có delay, logging và xử lý lỗi.
11. Áp dụng nguyên tắc đạo đức: terms of use, robots.txt, dữ liệu công khai và rate limiting.

<a id="setup"></a>
## Chuẩn bị môi trường

Các thư viện chính:

- `requests`: gửi HTTP request.
- `beautifulsoup4`: parse HTML/XML.
- `pandas`: lưu và phân tích bảng dữ liệu.
- `lxml`: hỗ trợ XPath.
- `selenium`: tự động điều khiển trình duyệt.

Một số phần có thể cần Internet. Notebook đã có ví dụ HTML/XML offline để sinh viên vẫn học được khi website gốc thay đổi.

In [15]:
import importlib.util
import subprocess
import sys
import pandas as pd

packages = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "lxml": "lxml",
    "selenium": "selenium"
}

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        print(f"Đang cài {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Môi trường đã sẵn sàng.")

Môi trường đã sẵn sàng.


In [2]:
from pathlib import Path

candidates = [
    Path.cwd() / "data",
    Path("/content/data"),
    Path.cwd().parent / "data"
]
DATA_DIR = next((p for p in candidates if p.exists()), None)

print("DATA_DIR =", DATA_DIR)
if DATA_DIR:
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = /content/data
 - HistoricalQuotes_2008-2018_googl.csv
 - cm_vote_20181010.xml
 - data.xml
 - xpath_allbags.png
 - xpath_onebag.png


<a id="intro"></a>
## 1. Web scraping là gì?

Web scraping là **xây dựng một agent tự động tải, parse và tổ chức dữ liệu từ web**.

Các tên gọi gần nghĩa:

- web scraping;
- screen scraping;
- web harvesting;
- web data extraction;
- web data mining.

### Vì sao cần web scraping?

- Web là nguồn dữ liệu phong phú.
- Tự động nhanh hơn thu thập thủ công.
- Hữu ích khi không có API hoặc API bị giới hạn.

### Một quy trình cơ bản

```text
URL
  ↓
HTTP Request
  ↓
HTML/XML/JSON Response
  ↓
Parser
  ↓
Trích xuất dữ liệu
  ↓
DataFrame
  ↓
CSV / Database / Analysis
```

<a id="http"></a>
## 2. HTTP và cơ chế Request/Response

HTTP là giao thức ở tầng ứng dụng được trình duyệt và web server sử dụng để trao đổi tài liệu web.

Khi người dùng mở một URL:

1. client tạo kết nối;
2. gửi request, thường là `GET`;
3. server trả response;
4. browser parse/render nội dung.

Trong Python, thư viện `requests` giúp ta thực hiện công việc tương tự trình duyệt ở mức HTTP.

In [3]:
import requests

url = "https://example.com"

try:
    response = requests.get(url, timeout=10)
    print("Status code:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
    print("Độ dài nội dung:", len(response.text))
except requests.RequestException as e:
    print("Không thể truy cập Internet trong môi trường hiện tại:", e)

Status code: 200
Content-Type: text/html; charset=utf-8
Độ dài nội dung: 577


### 2.1. Một số HTTP status code nên biết

- `200`: thành công.
- `301/302`: redirect.
- `403`: bị từ chối.
- `404`: không tìm thấy.
- `429`: gửi quá nhiều request.
- `500`: lỗi phía server.

Crawler nên kiểm tra status code trước khi parse.

In [4]:
def fetch_page(url, timeout=10):
    try:
        r = requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent": "Educational-Web-Scraping-Lab/1.0"}
        )
        r.raise_for_status()
        return r
    except requests.RequestException as e:
        print("Request error:", e)
        return None

r = fetch_page("https://example.com")
if r:
    print(r.status_code, len(r.text))

200 577


<a id="html"></a>
## 3. HTML và CSS cơ bản

HTML dùng để **cấu trúc** trang web.

Các tag thường gặp:

- `<h1>` ... `<h6>`: tiêu đề.
- `<p>`: đoạn văn.
- `<div>`: nhóm phần tử.
- `<a href="...">`: hyperlink.
- `<ul>`, `<ol>`, `<li>`: danh sách.
- `<table>`, `<tr>`, `<th>`, `<td>`: bảng.
- `<img src="...">`: hình ảnh.

CSS dùng để định dạng phần trình bày.

Hai khái niệm quan trọng cho scraping:

- `class`: có thể xuất hiện ở nhiều phần tử.
- `id`: thường dùng cho một phần tử duy nhất.

In [5]:
html_sample = '''
<html>
  <head><title>Ví dụ Web Scraping</title></head>
  <body>
    <h1 id="main-title">Danh sách sản phẩm</h1>
    <div class="product">
      <a href="/p1">Laptop A</a>
      <span class="price">1200</span>
    </div>
    <div class="product">
      <a href="/p2">Laptop B</a>
      <span class="price">1500</span>
    </div>
  </body>
</html>
'''

print(html_sample)


<html>
  <head><title>Ví dụ Web Scraping</title></head>
  <body>
    <h1 id="main-title">Danh sách sản phẩm</h1>
    <div class="product">
      <a href="/p1">Laptop A</a>
      <span class="price">1200</span>
    </div>
    <div class="product">
      <a href="/p2">Laptop B</a>
      <span class="price">1500</span>
    </div>
  </body>
</html>



<a id="ethics"></a>
## 4. Nguyên tắc pháp lý và đạo đức

Slide nhấn mạnh rằng trước khi scrape cần:

- kiểm tra **Terms of Use**;
- kiểm tra `robots.txt`;
- ưu tiên dữ liệu công khai;
- xin quyền/API nếu phù hợp;
- không gửi request quá nhanh;
- minh bạch mục đích thu thập.

## Checklist trước khi scrape

1. Dữ liệu có thực sự cần cho bài toán không?
2. Dữ liệu có công khai không?
3. Website có API chính thức không?
4. Điều khoản sử dụng cho phép gì?
5. Có thông tin cá nhân/nhạy cảm không?
6. Có cần login hoặc bypass biện pháp bảo vệ không?
7. Bao nhiêu trang và bao nhiêu request?
8. Delay giữa các request là bao nhiêu?

<a id="requests"></a>
## 5. Requests: lấy dữ liệu từ web

Ví dụ cơ bản:

In [6]:
import requests

url = "https://example.com"
headers = {
    "User-Agent": "Mozilla/5.0 Educational Lab",
    "Accept-Language": "en-US,en;q=0.5"
}

try:
    source = requests.get(url, headers=headers, timeout=10)
    print("Status:", source.status_code)
    print(source.text[:500])
except requests.RequestException as e:
    print(e)

Status: 200
<!doctype html><html lang=en><head><meta charset=utf-8><link rel=icon href=data:,><meta name=viewport content="width=device-width,initial-scale=1"><title>Example Domain</title><style>html{color-scheme:light dark;background:light-dark(#eee,#222)}body{font:16px/1.6 system-ui,sans-serif;max-width:26em;margin:auto;padding:25vh 2em 2em;text-align:center}</style></head><body><p>This domain is for use in documentation examples without needing permission. This is not a service; avoid relying on it for t


### 5.1. Parameters thay vì nối chuỗi URL

Khi website nhận tham số query, nên dùng `params`.

In [7]:
params = {"year": 2008}

try:
    demo = requests.get(
        "https://httpbin.org/get",
        params=params,
        timeout=10
    )
    if demo.ok:
        print("Final URL:", demo.url)
        print(demo.json())
except requests.RequestException as e:
    print("Không truy cập được httpbin:", e)

Final URL: https://httpbin.org/get?year=2008
{'args': {'year': '2008'}, 'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate, br', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.32.4', 'X-Amzn-Trace-Id': 'Root=1-6ac8c754-1833d6cf43fa6d8d09b1ba3c'}, 'origin': '104.198.24.13', 'url': 'https://httpbin.org/get?year=2008'}


<a id="bs4"></a>
## 6. BeautifulSoup: đọc và tìm phần tử HTML

BeautifulSoup biến HTML thành cây đối tượng có thể tìm kiếm.

Các thao tác thường dùng:

- `find()`: phần tử đầu tiên.
- `find_all()`: tất cả phần tử phù hợp.
- `select()`: CSS selector.
- `.text` / `.get_text()`: lấy text.
- `.get("href")`: lấy attribute.

In [8]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_sample, "html.parser")

print("Title:", soup.title.text)
print("H1:", soup.find("h1").get_text(strip=True))
print("Main ID:", soup.find(id="main-title").text)

Title: Ví dụ Web Scraping
H1: Danh sách sản phẩm
Main ID: Danh sách sản phẩm


### 6.1. `find_all()`

In [9]:
products = soup.find_all("div", class_="product")

for p in products:
    name = p.find("a").get_text(strip=True)
    href = p.find("a").get("href")
    price = p.find("span", class_="price").get_text(strip=True)
    print(name, href, price)

Laptop A /p1 1200
Laptop B /p2 1500


### 7.2. CSS Selector

In [10]:
for p in soup.select("div.product"):
    print(
        p.select_one("a").get_text(strip=True),
        p.select_one(".price").get_text(strip=True)
    )

Laptop A 1200
Laptop B 1500


<a id="single"></a>
## 7. Scraping một trang

Phần code đính kèm có ví dụ scrape **CityU News năm 2008** bằng:

- `requests`;
- BeautifulSoup;
- tìm table;
- lấy title, URL và date;
- đưa vào Pandas DataFrame.

Website gốc có thể đã thay đổi từ thời điểm notebook gốc, vì vậy ta chia thành:

1. mô phỏng offline để học đúng logic;
2. code live tham khảo có xử lý lỗi.

### 7.1. Ví dụ offline mô phỏng CityU News

In [11]:
cityu_html = '''
<table class="cityu-width-full">
  <tr>
    <td><a href="news1.html">Research Seminar 2008</a></td>
    <td><div>News</div><div>2008-01-15</div></td>
  </tr>
  <tr>
    <td><a href="news2.html">Student Award</a></td>
    <td><div>News</div><div>2008-03-22</div></td>
  </tr>
</table>
'''

page_html = BeautifulSoup(cityu_html, "html.parser")
news_container = page_html.find("table", class_="cityu-width-full")

titles, urls, dates = [], [], []

for row in news_container.find_all("tr"):
    a = row.find("a")
    divs = row.find_all("div")

    titles.append(a.get_text(strip=True) if a else None)
    urls.append(a.get("href") if a else None)
    dates.append(divs[-1].get_text(strip=True) if divs else None)

print(titles)
print(urls)
print(dates)

['Research Seminar 2008', 'Student Award']
['news1.html', 'news2.html']
['2008-01-15', '2008-03-22']


In [ ]:
import pandas as pd

cityu_2008 = pd.DataFrame({
    "title": titles,
    "url": urls,
    "date": dates
})

cityu_2008

### 7.2. Có kiểm tra lỗi

Code dưới đây giữ logic đó nhưng tránh lỗi nếu cấu trúc website hiện tại đã thay đổi.

In [12]:
from time import sleep
from random import randint

def scrape_cityu_year(year):
    url = f"http://www.cityu.edu.hk/com/News_News.aspx?year={year}"
    headers = {"Accept-Language": "en-US,en;q=0.5"}

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Lỗi khi tải {year}:", e)
        return pd.DataFrame(columns=["title", "url", "date"])

    sleep(randint(1, 3))
    soup = BeautifulSoup(response.content, "html.parser")
    container = soup.find("table", class_="cityu-width-full")

    if container is None:
        print("Không tìm thấy table theo cấu trúc notebook gốc.")
        return pd.DataFrame(columns=["title", "url", "date"])

    records = []
    for row in container.find_all("tr"):
        a = row.find("a")
        if not a:
            continue

        date_nodes = row.select("div:nth-of-type(2)")
        date_text = date_nodes[0].get_text(" ", strip=True) if date_nodes else None

        records.append({
            "title": a.get_text(" ", strip=True),
            "url": a.get("href"),
            "date": date_text
        })

    return pd.DataFrame(records)

# Chỉ chạy nếu muốn thử website live:
# df_2008_live = scrape_cityu_year(2008)
# df_2008_live.head()

<a id="multiple"></a>
## 8. Scraping nhiều trang

Notebook đính kèm mở rộng từ một năm sang giai đoạn **2008-2018**.

Ý tưởng:

```text
for year in years:
    tải trang
    delay
    parse HTML
    trích xuất record
    append kết quả
```

Không nên dùng `except:` trống vì dễ che mất lỗi.  
Nên xử lý exception cụ thể và lưu trạng thái.

In [17]:
def scrape_cityu_years(start_year=2008, end_year=2018):
    frames = []

    for year in range(start_year, end_year + 1):
        print("Đang xử lý:", year)
        df = scrape_cityu_year(year)

        if not df.empty:
            df["year"] = year
            frames.append(df)

    if not frames:
        return pd.DataFrame(columns=["title", "url", "date", "year"])

    return pd.concat(frames, ignore_index=True)

# Có thể mất thời gian và website gốc có thể thay đổi:
# cityu_10y = scrape_cityu_years()
# cityu_10y.head()

### 8.1. Lưu dữ liệu

In [18]:
# cityu_10y.to_csv("cityu_news_2008_2018.csv", index=False, encoding="utf-8-sig")

## 8.2. Thách thức mở rộng

1. Truy cập từng URL tin và lấy nội dung bài viết.
2. Lưu `title`, `date`, `url`, `article_text`.
3. Thêm cột `status_code`.
4. Ghi log URL lỗi.

<a id="crawler"></a>
## 9. Thiết kế một crawler nhỏ

Lưu ý:

- harvest nhiều trang;
- dùng function;
- xác định rõ **“What do I need?”** trước khi crawl.

Một crawler tốt nên tách các bước:

```text
fetch()
parse()
extract()
normalize()
save()
```

In [19]:
from urllib.parse import urljoin
import time

def fetch_html(url, delay=1):
    time.sleep(delay)
    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Educational-Crawler/1.0"},
            timeout=15
        )
        r.raise_for_status()
        return r.text
    except requests.RequestException as e:
        print("Fetch failed:", url, e)
        return None

def extract_links(html, base_url=None):
    soup = BeautifulSoup(html, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if base_url:
            href = urljoin(base_url, href)
        links.append({
            "text": a.get_text(" ", strip=True),
            "url": href
        })
    return links

demo_links = extract_links(html_sample, "https://example.org")
demo_links

[{'text': 'Laptop A', 'url': 'https://example.org/p1'},
 {'text': 'Laptop B', 'url': 'https://example.org/p2'}]

### 9.1. Thu thập có giới hạn

Không nên để crawler chạy vô hạn. Hãy đặt:

- `max_pages`;
- delay;
- danh sách visited;
- chỉ crawl cùng domain nếu bài toán yêu cầu.

In [20]:
from collections import deque
from urllib.parse import urlparse

def same_domain(url1, url2):
    return urlparse(url1).netloc == urlparse(url2).netloc

def crawl(start_url, max_pages=5, delay=1):
    queue = deque([start_url])
    visited = set()
    results = []

    while queue and len(visited) < max_pages:
        url = queue.popleft()
        if url in visited:
            continue

        visited.add(url)
        html = fetch_html(url, delay=delay)
        if html is None:
            continue

        results.append({
            "url": url,
            "html_length": len(html)
        })

        for item in extract_links(html, url):
            nxt = item["url"]
            if same_domain(start_url, nxt) and nxt not in visited:
                queue.append(nxt)

    return pd.DataFrame(results)

# Ví dụ live:
# crawl("https://example.com", max_pages=2)

<a id="datasource"></a>
## Dữ liệu có cấu trúc và không cấu trúc

### Structured data

Có cấu trúc hàng/cột hoặc schema rõ ràng:

- CSV
- Excel
- JSON
- XML
- database table

### Unstructured / less-structured data

- PDF
- nội dung văn bản tự do
- HTML nhìn từ góc độ nghiên cứu có thể cần parse lại để thành bảng.

Khi scrape, mục tiêu thường là biến HTML thành **structured DataFrame**.

<a id="xml"></a>
## 10. XML

XML biểu diễn dữ liệu theo cấu trúc cây.

Các thao tác trong code đính kèm:

- `ET.fromstring()`
- `ET.parse()`
- `.find()`
- `.findall()`
- `.iter()`
- `.get()`

In [21]:
import xml.etree.ElementTree as ET

xml_text = '''
<person>
  <name>Chuck</name>
  <phone type="intl">+1 734 303 4456</phone>
  <email hide="yes"/>
</person>
'''

tree = ET.fromstring(xml_text)

print("Name:", tree.find("name").text)
print("Phone:", tree.find("phone").text.strip())
print("Hide:", tree.find("email").get("hide"))

Name: Chuck
Phone: +1 734 303 4456
Hide: yes


### 10.1. XML nhiều phần tử

In [22]:
xml_users = '''
<stuff>
  <users>
    <user x="2"><id>001</id><name>Chuck</name></user>
    <user x="7"><id>009</id><name>Brent</name></user>
  </users>
</stuff>
'''

root = ET.fromstring(xml_users)
users = root.findall("users/user")

for user in users:
    print(
        user.find("id").text,
        user.find("name").text,
        user.get("x")
    )

001 Chuck 2
009 Brent 7


### 10.2. Đọc file XML đi kèm

In [23]:
if DATA_DIR is not None:
    xml_path = DATA_DIR / "data.xml"
    tree = ET.parse(xml_path)
    root = tree.getroot()

    for child in root:
        print(child.tag, child.attrib)

country {'name': 'Liechtenstein'}
country {'name': 'Singapore'}
country {'name': 'Panama'}


### 10.3. Hong Kong LegCo voting record

File XML này có trong code đính kèm.  
Ta trích `name-ch` và `vote` thành DataFrame.

In [24]:
if DATA_DIR is not None:
    vote_path = DATA_DIR / "cm_vote_20181010.xml"

    vote_tree = ET.parse(vote_path)
    vote_root = vote_tree.getroot()

    records = []
    for member in vote_root.iter("member"):
        vote_node = member.find("vote")
        records.append({
            "name": member.get("name-ch"),
            "vote": vote_node.text if vote_node is not None else None
        })

    vote_df = pd.DataFrame(records)
    print(vote_df.info())
    display(vote_df.head(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    68 non-null     object
 1   vote    68 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB
None


,name,vote
0,梁君彥,Present
1,涂謹申,Yes
2,梁耀忠,Yes
3,石禮謙,No
4,張宇人,No
5,李國麟,Absent
6,林健鋒,No
7,黃定光,Absent
8,李慧琼,Absent
9,陳克勤,Yes


### 10.4. Phân tích sơ bộ voting

In [25]:
if 'vote_df' in globals() and not vote_df.empty:
    display(vote_df["vote"].value_counts(dropna=False))

,count
vote,
Yes,24
No,19
Absent,17
Abstain,6
Present,2


<a id="json"></a>
## 11. JSON

JSON thường ánh xạ tự nhiên sang:

- `dict`;
- `list`;
- nested dict/list.

Python sử dụng module `json`.

In [26]:
import json

json_text = '''
{
  "name": "Chuck",
  "phone": {
    "type": "intl",
    "number": "+1 734 303 4456"
  },
  "email": {
    "hide": "yes"
  }
}
'''

info = json.loads(json_text)

print("Name:", info["name"])
print("Hide:", info["email"]["hide"])

Name: Chuck
Hide: yes


### 11.1. JSON dạng list

In [27]:
json_list = '''
[
  {"id": "001", "x": "2", "name": "Chuck"},
  {"id": "009", "x": "7", "name": "Brent"}
]
'''

users = json.loads(json_list)

for item in users:
    print(item["id"], item["name"], item["x"])

001 Chuck 2
009 Brent 7


### 11.2. Chuyển JSON nested thành DataFrame

Đây là bước quan trọng khi khai thác API hoặc social media data.

In [28]:
nested_json = [
    {
        "created_at": "2026-10-01",
        "text": "Data analytics is useful",
        "user": {
            "screen_name": "user_a",
            "followers_count": 120
        }
    },
    {
        "created_at": "2026-10-02",
        "text": "Learning web scraping",
        "user": {
            "screen_name": "user_b",
            "followers_count": 250
        }
    }
]

json_df = pd.json_normalize(nested_json)
json_df

,created_at,text,user.screen_name,user.followers_count
0,2026-10-01,Data analytics is useful,user_a,120
1,2026-10-02,Learning web scraping,user_b,250


### 11.3. Truy xuất an toàn field có thể thiếu


In [29]:
tweet = {
    "created_at": "2026-10-09",
    "user": {
        "screen_name": "demo_user",
        "followers_count": 123
    }
}

timestamp = tweet.get("created_at", "No timestamp")
followers = tweet.get("user", {}).get("followers_count", "No followers")

print(timestamp)
print(followers)

2026-10-09
123


<a id="xpath"></a>
## 12. XPath

XPath mô tả vị trí phần tử trong cây HTML/XML.

Ví dụ:

```text
/html/body/div
//*[@id="searchTerm"]
//div[@class="product"]
```

XPath được dùng để tìm:

- container sản phẩm;
- tên sản phẩm;
- mô tả;
- giá.

In [30]:
from lxml import html

tree = html.fromstring(html_sample)

names = tree.xpath('//div[@class="product"]/a/text()')
prices = tree.xpath('//div[@class="product"]/span[@class="price"]/text()')

print(names)
print(prices)

['Laptop A', 'Laptop B']
['1200', '1500']


### 12.1. So sánh CSS Selector và XPath

BeautifulSoup:

```python
soup.select("div.product a")
```

lxml XPath:

```python
tree.xpath('//div[@class="product"]/a')
```

Không có lựa chọn luôn tốt nhất. Chọn selector dễ đọc và ổn định với cấu trúc trang.

<a id="selenium"></a>
## 13. Selenium

Dùng Selenium cho **advanced automated data collection**.

Selenium phù hợp khi:

- nội dung được render bằng JavaScript;
- cần click/scroll/search;
- trang thay đổi sau tương tác;
- Requests không lấy được dữ liệu cuối cùng.

In [31]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

def create_headless_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=options)

# Chỉ tạo driver khi máy đã có Chrome/Chromium:
# driver = create_headless_driver()

### 15.1. Selenium với trang HTML đơn giản

Ví dụ này là template; cần browser/driver tương thích trên máy.

In [32]:
selenium_template = '''
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://example.com")

title = driver.find_element(By.TAG_NAME, "h1").text
links = driver.find_elements(By.TAG_NAME, "a")

print(title)
for link in links:
    print(link.text, link.get_attribute("href"))

driver.quit()
'''

print(selenium_template)


from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://example.com")

title = driver.find_element(By.TAG_NAME, "h1").text
links = driver.find_elements(By.TAG_NAME, "a")

print(title)
for link in links:
    print(link.text, link.get_attribute("href"))

driver.quit()



### 13.2. Mẫu XPath sản phẩm

Code Selfridges gốc lặp qua nhiều sản phẩm, lấy:

- `name`;
- `desc`;
- `price`;

sau đó đưa vào DataFrame.

Vì cấu trúc Selfridges hiện tại có thể khác hoàn toàn, ta minh họa cấu trúc code thay vì chạy cứng XPath cũ.

In [33]:
def scrape_products_with_selenium(driver, product_xpath, name_xpath, desc_xpath, price_xpath):
    rows = []

    products = driver.find_elements(By.XPATH, product_xpath)

    for product in products:
        def safe_text(xpath):
            try:
                return product.find_element(By.XPATH, xpath).text.strip()
            except Exception:
                return None

        rows.append({
            "name": safe_text(name_xpath),
            "description": safe_text(desc_xpath),
            "price": safe_text(price_xpath)
        })

    return pd.DataFrame(rows)

# Ví dụ selector tương đối, phải điều chỉnh theo website thực tế:
# df = scrape_products_with_selenium(
#     driver,
#     '//div[contains(@class,"product")]',
#     './/a[contains(@class,"name")]',
#     './/div[contains(@class,"description")]',
#     './/*[contains(@class,"price")]'
# )

### 13.3. Khi nào chọn Requests/BeautifulSoup và khi nào chọn Selenium?

| Tình huống | Công cụ ưu tiên |
|---|---|
| HTML tĩnh, dữ liệu có sẵn trong response | Requests + BeautifulSoup |
| API JSON/XML | Requests + json/XML parser |
| Table HTML đơn giản | Pandas `read_html()` hoặc BeautifulSoup |
| Nội dung sinh bởi JavaScript | Selenium |
| Cần click/search/scroll | Selenium |
| Cần tốc độ cao với nhiều trang tĩnh | Requests |

<a id="project"></a>
## BÀI MỞ RỘNG - Web Data Collection Pipeline

### Bài toán

Xây dựng một pipeline thu thập dữ liệu tin/bài viết công khai.

### Output mong muốn

| field | ý nghĩa |
|---|---|
| title | tiêu đề |
| date | ngày |
| url | URL |
| content | nội dung |
| source | nguồn |
| collected_at | thời điểm thu thập |
| status | trạng thái |

### Các bước

1. Xác định nguồn dữ liệu.
2. Kiểm tra điều khoản/robots/API.
3. Xác định cấu trúc HTML.
4. Viết `fetch_page()`.
5. Viết `parse_list_page()`.
6. Viết `parse_detail_page()`.
7. Thêm delay.
8. Xử lý lỗi.
9. Chuẩn hóa dữ liệu.
10. Lưu CSV.

In [34]:
from datetime import datetime

def normalize_record(record):
    return {
        "title": (record.get("title") or "").strip(),
        "date": record.get("date"),
        "url": record.get("url"),
        "content": (record.get("content") or "").strip(),
        "source": record.get("source"),
        "collected_at": datetime.now().isoformat(timespec="seconds"),
        "status": record.get("status", "ok")
    }

demo_record = normalize_record({
    "title": "  Sample News  ",
    "date": "2026-10-09",
    "url": "https://example.org/news/1",
    "content": " Demo content ",
    "source": "Example"
})

demo_record

{'title': 'Sample News',
 'date': '2026-10-09',
 'url': 'https://example.org/news/1',
 'content': 'Demo content',
 'source': 'Example',
 'collected_at': '2026-10-09T10:59:09',
 'status': 'ok'}

### Validation trước khi lưu

Kiểm tra:

- URL trùng;
- title rỗng;
- status code lỗi;
- ngày sai format;
- content quá ngắn.

In [35]:
def validate_dataframe(df):
    report = {
        "rows": len(df),
        "missing_title": int(df["title"].isna().sum()) if "title" in df else None,
        "missing_url": int(df["url"].isna().sum()) if "url" in df else None,
        "duplicated_url": int(df["url"].duplicated().sum()) if "url" in df else None
    }
    return report

sample_df = pd.DataFrame([demo_record])
validate_dataframe(sample_df)

{'rows': 1, 'missing_title': 0, 'missing_url': 0, 'duplicated_url': 0}

<a id="exercise"></a>
## Bài tập tự luyện

### Bài 1 - HTML

Cho một đoạn HTML gồm 10 sản phẩm. Hãy:

1. lấy tên;
2. lấy giá;
3. lấy link;
4. lưu vào DataFrame.

---

### Bài 2 - Requests + BeautifulSoup

Chọn một trang web công khai, tĩnh và được phép truy cập.

1. gửi GET request;
2. kiểm tra status code;
3. parse HTML;
4. trích ít nhất 3 trường;
5. lưu CSV.

---

### Bài 3 - Multi-page

Viết crawler cho tối đa 5 trang.

Yêu cầu:

- delay ít nhất 1 giây;
- không crawl URL trùng;
- lưu URL lỗi;
- không vượt `max_pages`.

---

### Bài 4 - XML

Với `cm_vote_20181010.xml`:

1. trích tên thành viên;
2. trích phiếu;
3. đếm số phiếu theo loại;
4. tính tỷ lệ phần trăm từng loại.

---

### Bài 5 - JSON

Tạo JSON nested gồm user + posts.

1. dùng `json.loads()`;
2. trích field nested;
3. dùng `pd.json_normalize()`;
4. lưu CSV.

---

### Bài 6 - XPath

Với HTML mẫu:

1. viết XPath lấy tất cả product;
2. XPath lấy tên;
3. XPath lấy giá;
4. so sánh với CSS selector.

---

### Bài 7 - Selenium

Chọn một trang demo hợp pháp cho browser automation.

1. mở trang;
2. tìm một textbox;
3. nhập từ khóa;
4. click search;
5. lấy 5 kết quả;
6. đóng driver.

## 17.1. Bài tập mở rộng theo code gốc

### Challenge 1
Từ danh sách CityU News, truy cập URL chi tiết và lấy nội dung bài viết.

### Challenge 2
Scrape một trang danh sách bài giảng/video và lưu:

- title;
- date;
- video URL.

### Challenge 3
Từ XML LegCo, phân tích pattern voting bằng Pandas.

### Challenge 4
Từ website thương mại điện tử phù hợp, phân tích:

- brand nào có giá trung bình cao nhất;
- brand nào có độ phân tán giá lớn nhất;
- sản phẩm nào giảm giá nhiều nhất.

<a id="summary"></a>
# Tổng kết

Trình tự chính:

```text
Web Technologies
   ↓
HTTP
   ↓
HTML / CSS
   ↓
Requests
   ↓
BeautifulSoup
   ↓
Single-page scraping
   ↓
Multi-page scraping
   ↓
Crawler
   ↓
XML / JSON
   ↓
XPath
   ↓
Selenium
   ↓
Structured DataFrame
```

## Những nguyên tắc cần nhớ

1. Trước tiên xác định **dữ liệu cần gì**, không phải scrape tất cả những gì nhìn thấy.
2. Ưu tiên API nếu có API phù hợp.
3. Với trang tĩnh, ưu tiên Requests + BeautifulSoup.
4. Chỉ dùng Selenium khi thực sự cần browser automation/JavaScript.
5. Không dùng `except:` trống.
6. Luôn đặt timeout.
7. Rate-limit request.
8. Không bypass login, CAPTCHA hay cơ chế bảo vệ.
9. Kiểm tra Terms of Use và robots.txt.
10. Lưu dữ liệu cùng metadata: URL, thời gian thu thập, status.